# Diurnal Pattern Exploration

Explores how diurnal DO, pH, and temperature patterns respond to:
- Time of year (day of year)
- Cloud cover
- Wind speed

Reference: Szyper et al. (1992). *Aquacultural Engineering*, 11(2), 73-89.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
from xoceania_sim import PondSimulator, SimConfig

## 1. Effect of cloud cover on diurnal DO

In [ ]:
import copy

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
cloud_values = [0.0, 0.3, 0.6, 0.9]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for cloud, color in zip(cloud_values, colors):
    cfg = SimConfig()
    cfg.weather.cloud_cover = cloud
    sim = PondSimulator(cfg)
    result = sim.run(t_span=(0, 48))
    df = result.df
    t = df.index
    axes[0].plot(t, df['DO'], color=color, lw=2, label=f'cloud={cloud:.0%}')
    axes[1].plot(t, df['pH'], color=color, lw=2, label=f'cloud={cloud:.0%}')

axes[0].set_ylabel('DO (mg/L)')
axes[0].set_ylim(2, 16)
axes[0].axhline(5, color='gray', ls=':', lw=1)
axes[0].legend(fontsize=9)

axes[1].set_ylabel('pH')
axes[1].set_ylim(6.5, 11)
axes[1].set_xlabel('Time (h)')
axes[1].set_xticks(range(0, 49, 6))

fig.suptitle('Effect of Cloud Cover on Diurnal DO and pH', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Effect of day of year (seasonality)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
doy_values = [50, 100, 150, 200, 250, 300]
palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(doy_values)))

for doy, color in zip(doy_values, palette):
    cfg = SimConfig()
    cfg.weather.day_of_year = doy
    sim = PondSimulator(cfg)
    result = sim.run(t_span=(0, 48))
    df = result.df
    t = df.index
    axes[0].plot(t, df['DO'], color=color, lw=1.5, label=f'DOY {doy}')
    axes[1].plot(t, df['T'], color=color, lw=1.5)

axes[0].set_ylabel('DO (mg/L)')
axes[0].legend(fontsize=8, ncol=2)
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_xlabel('Time (h)')
axes[1].set_xticks(range(0, 49, 6))
fig.suptitle('Seasonal Effect on DO and Temperature (DOY)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. DO range analysis across cloud conditions

In [ ]:
cloud_sweep = np.linspace(0.0, 0.95, 20)
do_max_vals, do_min_vals = [], []

for cloud in cloud_sweep:
    cfg = SimConfig()
    cfg.weather.cloud_cover = cloud
    sim = PondSimulator(cfg)
    result = sim.run(t_span=(0, 48))
    df = result.df
    # Second cycle only
    df2 = df[df.index >= 24]
    do_max_vals.append(df2['DO'].max())
    do_min_vals.append(df2['DO'].min())

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(cloud_sweep * 100, do_min_vals, do_max_vals, alpha=0.3, color='#1f77b4')
ax.plot(cloud_sweep * 100, do_max_vals, 'b-', lw=2, label='Daily max DO')
ax.plot(cloud_sweep * 100, do_min_vals, 'b--', lw=2, label='Daily min DO')
ax.axhline(5, color='red', ls=':', lw=1.5, label='5 mg/L stress threshold')
ax.set_xlabel('Cloud cover (%)')
ax.set_ylabel('DO (mg/L)')
ax.set_title('DO Range vs. Cloud Cover (48-h simulation, day 2)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()